In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Load ESA WorldCover
worldcover = ee.ImageCollection('ESA/WorldCover/v100').first()

# Barbados boundary from GAUL
barbados = (
    ee.FeatureCollection('FAO/GAUL/2015/level0')
    .filter(ee.Filter.eq('ADM0_NAME', 'Barbados'))
)

# Selected classes of interest
classes = [10, 20, 30, 40, 50, 60]

class_names = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare / sparse vegetation"
}

# Create mask for selected classes
mask = worldcover.remap(
    classes,              # from
    [1] * len(classes)    # to
).selfMask()

# Apply mask and clip
filtered = worldcover.updateMask(mask).clip(barbados)

# Visualization
vis_params = {
    'min': 10,
    'max': 60,
    'palette': [
        '006400',  # Tree cover (10)
        'ffbb22',  # Shrubland (20)
        'ffff4c',  # Grassland (30)
        'f096ff',  # Cropland (40)
        'fa0000',  # Built-up (50)
        'b4b4b4'   # Bare/sparse vegetation (60)
    ]
}

# Display
m = geemap.Map()
m.centerObject(barbados, 10)
m.addLayer(filtered, vis_params, 'Selected Land Cover')
# m.addLayer(barbados, {'color': 'black'}, 'Barbados')
m

task = ee.batch.Export.image.toDrive(
    image=filtered.rename('landcover'),
    description='Barbados_Landcover',
    folder='GEE_Exports',
    fileNamePrefix='barbados_landcover',
    region=barbados.geometry(),
    scale=10,  # ESA WorldCover resolution is 10 m
    maxPixels=1e13
)

task.start()